In [3]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)



In [4]:
df = pd.read_excel(
    "../data/HHS_Unaccompanied_Alien_Children_Program.xlsx"
)

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date")

df.head()

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
719,2023-01-12,33,53,34,6566,436
718,2023-01-22,32,49,39,7122,227
717,2023-01-23,32,50,39,7280,181
716,2023-01-24,47,42,47,7433,175
715,2023-01-25,20,22,41,7538,180


In [5]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

Rows: 720
Columns: 6
<class 'pandas.DataFrame'>
RangeIndex: 720 entries, 719 to 0
Data columns (total 6 columns):
 #   Column                                           Non-Null Count  Dtype         
---  ------                                           --------------  -----         
 0   Date                                             720 non-null    datetime64[us]
 1   Children apprehended and placed in CBP custody*  720 non-null    int64         
 2   Children in CBP custody                          720 non-null    int64         
 3   Children transferred out of CBP custody          720 non-null    int64         
 4   Children in HHS Care                             720 non-null    int64         
 5   Children discharged from HHS Care                720 non-null    int64         
dtypes: datetime64[us](1), int64(5)
memory usage: 33.9 KB


In [6]:
df.isnull().sum()

Date                                               0
Children apprehended and placed in CBP custody*    0
Children in CBP custody                            0
Children transferred out of CBP custody            0
Children in HHS Care                               0
Children discharged from HHS Care                  0
dtype: int64

In [7]:
missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": round(df.isnull().mean()*100,2)
})

missing

,Missing Count,Missing %
Date,0,0.0
Children apprehended and placed in CBP custody*,0,0.0
Children in CBP custody,0,0.0
Children transferred out of CBP custody,0,0.0
Children in HHS Care,0,0.0
Children discharged from HHS Care,0,0.0


In [8]:
df["Date"].duplicated().sum()

np.int64(0)

In [9]:
all_dates = pd.date_range(
    start=df["Date"].min(),
    end=df["Date"].max(),
    freq="D"
)

missing_dates = all_dates.difference(df["Date"])

print("Missing dates:", len(missing_dates))

Missing dates: 355


In [10]:
invalid_transfer = df[
    df["Children transferred out of CBP custody"]
    >
    df["Children in CBP custody"]
]

invalid_transfer.shape

(86, 6)

In [11]:
invalid_discharge = df[
    df["Children discharged from HHS Care"]
    >
    df["Children in HHS Care"]
]

invalid_discharge.shape

(0, 6)

In [12]:
df["Total_System_Load"] = (
    df["Children in CBP custody"]
    +
    df["Children in HHS Care"]
)

df["Net_Daily_Intake"] = (
    df["Children transferred out of CBP custody"]
    -
    df["Children discharged from HHS Care"]
)

df["Load_Growth_Rate"] = (
    df["Total_System_Load"]
    .pct_change()
    * 100
)

df["Backlog"] = (
    df["Net_Daily_Intake"]
    .cumsum()
)

In [13]:
df.describe()

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care,Total_System_Load,Net_Daily_Intake,Load_Growth_Rate,Backlog
count,720,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,719.000000,720.000000
mean,2024-07-06 05:30:00,93.523611,171.494444,128.668056,6061.275000,173.406944,6232.769444,-44.738889,-0.116894,-25888.534722
min,2023-01-12 00:00:00,0.000000,7.000000,0.000000,1972.000000,0.000000,2002.000000,-465.000000,-11.601513,-33908.000000
25%,2023-10-16 18:00:00,12.000000,36.000000,14.000000,2467.750000,19.750000,2500.750000,-94.250000,-0.689345,-32173.750000
50%,2024-07-05 12:00:00,99.000000,193.000000,157.000000,6406.500000,181.000000,6633.500000,-11.500000,0.123877,-31494.000000
75%,2025-03-25 06:00:00,147.250000,263.250000,199.250000,8010.250000,267.000000,8249.500000,5.250000,0.858314,-20965.750000
max,2025-12-21 00:00:00,333.000000,531.000000,440.000000,11516.000000,505.000000,11762.000000,206.000000,8.339628,-402.000000
std,NaN,72.646625,126.354965,97.322012,2833.070109,125.702841,2918.458078,95.863856,1.895490,9665.680327


In [14]:
fig = px.line(
    df,
    x="Date",
    y="Total_System_Load",
    title="Total System Load Over Time"
)

fig.show()


In [15]:
import sys

print(sys.executable)

/Users/chandini/Creative Cloud Files/WorkSpace /unified mentor project /UAC_Care_Load_Analytics/.venv/bin/python


In [16]:
import nbformat
print(nbformat.__version__)

5.10.4


In [17]:
df["Load_7D"] = (
    df["Total_System_Load"]
    .rolling(7)
    .mean()
)

df["Load_14D"] = (
    df["Total_System_Load"]
    .rolling(14)
    .mean()
)

In [18]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df["Date"],
        y=df["Total_System_Load"],
        name="Daily Load"
    )
)

fig.add_trace(
    go.Scatter(
        x=df["Date"],
        y=df["Load_7D"],
        name="7-Day Avg"
    )
)

fig.add_trace(
    go.Scatter(
        x=df["Date"],
        y=df["Load_14D"],
        name="14-Day Avg"
    )
)

fig.show()

In [19]:
df.to_csv(
    "../data/uac_processed.csv",
    index=False
)

In [20]:
# Latest Values

latest_load = df["Total_System_Load"].iloc[-1]

avg_net_intake = df["Net_Daily_Intake"].mean()

volatility = df["Total_System_Load"].std()

backlog_rate = df["Backlog"].iloc[-1] / len(df)

discharge_offset_ratio = (
    df["Children discharged from HHS Care"].sum()
    /
    df["Children transferred out of CBP custody"].sum()
)

print("Latest System Load:", round(latest_load))
print("Average Net Intake:", round(avg_net_intake,2))
print("Volatility Index:", round(volatility,2))
print("Backlog Rate:", round(backlog_rate,2))
print("Discharge Offset Ratio:", round(discharge_offset_ratio,2))

Latest System Load: 2502
Average Net Intake: -44.74
Volatility Index: 2918.46
Backlog Rate: -44.74
Discharge Offset Ratio: 1.35


In [21]:
stress_threshold = df["Total_System_Load"].quantile(0.90)

stress_threshold

np.float64(9920.9)

In [22]:
df["Stress_Flag"] = (
    df["Total_System_Load"] >= stress_threshold
)

In [23]:
df["Stress_Flag"].value_counts()

Stress_Flag
False    648
True      72
Name: count, dtype: int64

In [24]:
stress_days = (
    df[df["Stress_Flag"]]
    .sort_values(
        "Total_System_Load",
        ascending=False
    )
)

stress_days.head(10)

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care,Total_System_Load,Net_Daily_Intake,Load_Growth_Rate,Backlog,Load_7D,Load_14D,Stress_Flag
495,2023-12-20,136,246,123,11516,390,11762,-267,2.492158,-28928,11409.285714,11211.571429,True
494,2023-12-21,128,228,177,11375,497,11603,-320,-1.351811,-29248,11459.714286,11267.928571,True
496,2023-12-19,105,185,158,11291,272,11476,-114,1.226074,-28661,11323.000000,11132.857143,True
499,2023-12-14,150,157,201,11242,396,11399,-195,0.228612,-28016,11210.142857,10915.357143,True
558,2023-09-19,133,287,189,11095,260,11382,-71,1.362543,-16907,11079.428571,10800.285714,True
557,2023-09-20,187,310,247,11064,364,11374,-117,-0.070286,-17024,11170.714286,10852.214286,True
500,2023-12-13,116,181,182,11192,335,11373,-153,1.093333,-27821,11151.857143,10818.357143,True
556,2023-09-21,168,285,210,11067,428,11352,-218,-0.193424,-17242,11226.142857,10918.785714,True
493,2023-12-25,90,196,205,11144,347,11340,-142,-2.266655,-29390,11455.000000,11303.428571,True
490,2023-12-28,68,107,102,11230,357,11337,-255,0.247590,-30120,11446.571429,11357.714286,True


In [25]:
#Monthly Trend Analysis
monthly = (
    df.set_index("Date")
      .resample("ME")
      .agg({
          "Total_System_Load":"mean",
          "Net_Daily_Intake":"mean",
          "Children in CBP custody":"mean",
          "Children in HHS Care":"mean"
      })
      .reset_index()
)

monthly.head()

,Date,Total_System_Load,Net_Daily_Intake,Children in CBP custody,Children in HHS Care
0,2023-01-31,7413.500000,-197.500000,43.875000,7369.625000
1,2023-02-28,7967.105263,-134.000000,171.315789,7795.789474
2,2023-03-31,8190.473684,-104.000000,245.105263,7945.368421
3,2023-04-30,8092.857143,-81.000000,250.190476,7842.666667
4,2023-05-31,8404.090909,-119.727273,216.954545,8187.136364


In [26]:
#Monthly System Load Chart
fig = px.line(
    monthly,
    x="Date",
    y="Total_System_Load",
    title="Monthly Average System Load"
)

fig.show()

In [27]:
#Monthly Intake Pressure
fig = px.bar(
    monthly,
    x="Date",
    y="Net_Daily_Intake",
    title="Monthly Net Intake Pressure"
)

fig.show()

In [28]:
#Create Final Processed Dataset
analytics_df = df.copy()

analytics_df.to_csv(
    "../data/uac_analytics_dataset.csv",
    index=False
)

In [29]:
analytics_df.shape

(720, 13)

In [ ]:
# Peak Load

peak_load = df["Total_System_Load"].max()

peak_date = df.loc[
    df["Total_System_Load"].idxmax(),
    "Date"
]

print("Peak Load:", peak_load)
print("Peak Date:", peak_date)

In [30]:
df.sort_values(
    "Net_Daily_Intake",
    ascending=False
)[
    ["Date", "Net_Daily_Intake"]
].head(10)

,Date,Net_Daily_Intake
460,2024-02-12,206
459,2024-02-13,189
410,2024-04-23,173
456,2024-02-19,153
409,2024-04-24,153
384,2024-05-30,149
464,2024-02-06,145
258,2024-12-03,135
328,2024-08-20,133
445,2024-03-05,131


In [31]:
df.sort_values(
    "Children discharged from HHS Care",
    ascending=False
)[
    ["Date",
     "Children discharged from HHS Care"]
].head(10)

,Date,Children discharged from HHS Care
570,2023-08-31,505
494,2023-12-21,497
550,2023-10-01,493
574,2023-08-27,488
481,2024-01-11,476
504,2023-12-07,466
498,2023-12-17,456
536,2023-10-22,454
551,2023-09-28,447
575,2023-08-24,443


In [32]:
monthly_summary = (
    df.set_index("Date")
      .resample("ME")
      .agg({
          "Total_System_Load":"mean",
          "Net_Daily_Intake":"mean",
          "Children in CBP custody":"mean",
          "Children in HHS Care":"mean",
          "Children discharged from HHS Care":"sum"
      })
)

monthly_summary.head()

,Total_System_Load,Net_Daily_Intake,Children in CBP custody,Children in HHS Care,Children discharged from HHS Care
Date,,,,,
2023-01-31,7413.500000,-197.500000,43.875000,7369.625000,1856
2023-02-28,7967.105263,-134.000000,171.315789,7795.789474,5183
2023-03-31,8190.473684,-104.000000,245.105263,7945.368421,5375
2023-04-30,8092.857143,-81.000000,250.190476,7842.666667,5668
2023-05-31,8404.090909,-119.727273,216.954545,8187.136364,6256


In [33]:
monthly_summary.to_csv(
    "../reports/monthly_summary.csv"
)

In [34]:
print("Peak Load:", df["Total_System_Load"].max())

print("Average Load:", round(df["Total_System_Load"].mean(),2))

print("Average Net Intake:", round(df["Net_Daily_Intake"].mean(),2))

print("Peak HHS Care:", df["Children in HHS Care"].max())

print("Peak CBP Custody:", df["Children in CBP custody"].max())

Peak Load: 11762
Average Load: 6232.77
Average Net Intake: -44.74
Peak HHS Care: 11516
Peak CBP Custody: 531


In [ ]:
df.sort_values(
    "Total_System_Load",
    ascending=False
)[
    ["Date","Total_System_Load"]
].head(10)